<a href="https://colab.research.google.com/github/omzeybek/Yeditepe_Data_Science_DATS501/blob/main/Week12_XGboost_Example_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Classifier Algorithm

## Introduction to XGBoost

XGBoost (eXtreme Gradient Boosting) is an optimized distributed gradient boosting library designed to be highly efficient, flexible, and portable.

It implements machine learning algorithms under the Gradient Boosting framework.

**Key Features:**

*   **Scalability:** Runs on a single machine or in distributed environments (Hadoop, Spark).

*   **Efficiency:** Achieves state-of-the-art results using less computation resources.

*   **Flexibility:** Supports various objective functions, including regression, classification, and ranking.

*   **Performance:** Known for its speed and predictive power, making it a popular choice for Kaggle competitions and real-world applications.

**How it works (briefly):**

XGBoost builds an ensemble of decision trees sequentially. Each new tree attempts to correct the errors of the previous ones. It uses a more regularized model formalization to control over-fitting, which gives it better performance. It also incorporates parallel processing, tree pruning, and handling of missing values natively.

## Import

In [ ]:
import xgboost as xgb #--> algortihm
from sklearn.datasets import load_breast_cancer # ---> sample data set from scikit learn
from sklearn.model_selection import train_test_split, cross_val_score # ---> Evaluation metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay # ---> Validation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

Load dataset (both target and regressors)

In [ ]:
cancer_data = load_breast_cancer()
X = pd.DataFrame(cancer_data.data, columns=cancer_data.feature_names)
y = cancer_data.target

## Data Preprocessing : Feature Engineering

Polynomial Feature: Emphasize larger tumor areas

In [ ]:
X['mean_area_squared'] = X['mean area'] ** 2

Interaction Feature: Ratio of perimeter to smoothness

In [ ]:
X['perimeter_smoothness_ratio'] = X['mean perimeter'] / (X['mean smoothness'] + 1e-5)

##Estimation

Now we need to split the training and validation datasets. While the training dataset will help us design our tree algorithm for an efficient solution, the validation data will serve as a testing ground to see if the trained model still holds for previously unseen data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

Inıtializating Xgboost classifier

In [ ]:
clf = xgb.XGBClassifier(eval_metric='logloss', random_state=42)

Peform 5-fold CV

In [ ]:
cv_scores = cross_val_score(clf, X_train, y_train, cv=5, scoring='accuracy')

In [ ]:
print(f"Mean CV Accuracy: {cv_scores.mean():.4f}\n")

Now as our model is robust enough to be used in estimation. We should score training with test set.

In [ ]:
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cancer_data.target_names)

In [ ]:
disp

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap=plt.cm.Blues, ax=ax)
plt.title("XGBoost Classification (with Feature Engineering)")

#Estimation Algorithm

In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
diabetes_data = load_diabetes()


In [ ]:
X = pd.DataFrame(diabetes_data.data, columns=diabetes_data.feature_names)
y = diabetes_data.target

In [ ]:
#Interaction Term: Compounded risk of BMI and Blood Pressure
X['bmi_bp_interaction'] = X['bmi'] * X['bp']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
reg = xgb.XGBRegressor(random_state=42)

###Hyperparameter tuning

Define base algortihm

In [ ]:
reg = xgb.XGBRegressor(random_state=42)

Now we need to specify values that should be tested to reach best algortihm.

In [ ]:
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [50, 100, 200]
}

In [ ]:
grid_search = GridSearchCV(
    estimator=reg,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1
)

In [ ]:
grid_search.fit(X_train, y_train)

Now store best_mode for later scoring efforts

In [ ]:
best_model = grid_search.best_estimator_

In [ ]:
print(f"Best Hyperparameters: {grid_search.best_params_}")

In [ ]:
y_pred = best_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [ ]:
print(f"\n--- Final Model Performance on Test Set ---")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R2): {r2:.4f}")